# Instance check — run this first

TC6035 Part 1. This notebook confirms the course library works, builds **your**
instance, draws it, and spends a few evaluations against the budget.

It is not graded. If every cell below runs, your environment is correct and you
can open `m0_foundations.ipynb`.

## 1. Make the library importable

In [ ]:
# In Google Colab: upload tc6035-starter.zip with the file pane, then
# uncomment the next line.
# !unzip -o tc6035-starter.zip

import sys, pathlib

try:
    import tc6035                      # already importable: nothing to do
    print(f"tc6035 found at {pathlib.Path(tc6035.__file__).parent}")
except ModuleNotFoundError:
    here = pathlib.Path.cwd()
    for base in (here, *here.parents):
        candidate = base / "src"
        if (candidate / "tc6035").is_dir():
            sys.path.insert(0, str(candidate))
            print(f"added {candidate} to sys.path")
            break
        candidate = base / "tc6035-starter" / "src"
        if (candidate / "tc6035").is_dir():
            sys.path.insert(0, str(candidate))
            print(f"added {candidate} to sys.path")
            break
    else:
        raise SystemExit(
            "Could not find the tc6035 package. Unzip tc6035-starter.zip into "
            "this folder, or set sys.path to point at its src/ directory.")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from tc6035 import build_instance, SensorPlacementProblem, BudgetExceeded

print("library imported")

## 2. Your identity

Replace `STUDENT_ID` with your own. `SALT` comes from the instance manifest on
Canvas and is almost always 0.

In [ ]:
STUDENT_ID = "A01234567"   # <-- yours
SALT       = 0             # <-- from the manifest

instance = build_instance(STUDENT_ID, salt=SALT)
spec = instance.spec

print(f"instance for {STUDENT_ID}, salt {SALT}")
print(f"  candidate sites : {spec.n_candidates}")
print(f"  demand points   : {instance.demand.size}")
print(f"  field           : {spec.field_size:.0f} x {spec.field_size:.0f}")
print(f"  power range     : [{spec.power_min}, {spec.power_max}]")
print(f"  per-site caps   : min {instance.power_cap.min():.1f}, "
      f"max {instance.power_cap.max():.1f}  <- heterogeneous")

### The instance is deterministic

The same ID always produces the same instance, on any machine. That is what lets
you pick the work up elsewhere and still be working on your problem.

In [ ]:
again = build_instance(STUDENT_ID, salt=SALT)
assert np.array_equal(instance.candidates, again.candidates)
assert np.array_equal(instance.demand, again.demand)
print("rebuilt identically")

other = build_instance("A09999999", salt=0)
print("a different ID gives a different instance:",
      not np.array_equal(instance.candidates, other.candidates))

## 3. Draw it\n\nSite size shows each site's power cap. Darker demand means heavier weight.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 7))
g = int(np.sqrt(instance.demand.size))
ax.imshow(instance.demand.reshape(g, g).T, origin="lower", cmap="Blues",
          extent=[0, spec.field_size, 0, spec.field_size], alpha=0.85)
ax.scatter(instance.candidates[:, 0], instance.candidates[:, 1],
           s=18 + 90 * (instance.power_cap / spec.power_max),
           facecolor="none", edgecolor="crimson", linewidth=1.3,
           label="candidate site (size = power cap)")
ax.set_title(f"Instance {STUDENT_ID}")
ax.set_xlabel("x"); ax.set_ylabel("y"); ax.legend(loc="upper right", fontsize=8)
plt.tight_layout(); plt.show()

## 4. Evaluate a solution

A solution is a pair: `active` (which sites are on) and `power` (how much each
one transmits). `evaluate_verbose` returns the breakdown.

In [ ]:
problem = SensorPlacementProblem(instance, budget=5000, seed=0)
rng = np.random.default_rng(0)

active = rng.random(spec.n_candidates) < 0.30
power  = rng.uniform(spec.power_min, spec.power_max, spec.n_candidates)

r = problem.evaluate_verbose(active, power)
print(f"objective     {r.objective:+.4f}   <- maximize this")
print(f"  coverage    {r.coverage:.4f}")
print(f"  energy      {r.energy:.4f}")
print(f"  interference {r.interference:.4f}")
print(f"  active      {r.n_active}")
print(f"  feasible    {r.feasible}   ({r.n_components} connected component(s))")
print(f"\nevaluations used: {problem.evaluations_used} / {problem.budget}")

### Feasibility

A deployment is feasible only when the active sensors form one connected
communication graph. Most random deployments are not, and the objective already
includes the penalty.

In [ ]:
feasible = 0
for _ in range(200):
    a = rng.random(spec.n_candidates) < 0.30
    p = rng.uniform(spec.power_min, spec.power_max, spec.n_candidates)
    feasible += problem.evaluate_verbose(a, p).feasible
print(f"{feasible} of 200 random deployments were feasible")
print(f"evaluations used: {problem.evaluations_used} / {problem.budget}")

## 5. The budget is enforced

Asking for more than the budget raises rather than returning a value. A batch of
`n` costs `n`, not one.

In [ ]:
probe = SensorPlacementProblem(instance, budget=10, seed=0)
try:
    for _ in range(11):
        probe.evaluate(active, power)
except BudgetExceeded as error:
    print(f"BudgetExceeded after {probe.evaluations_used} evaluations")
    print(f"  {error}")

In [ ]:
batch = SensorPlacementProblem(instance, budget=5000, seed=0)
values = batch.evaluate_batch(
    np.array([active, ~active]), np.array([power, power]))
print(f"a batch of 2 cost {batch.evaluations_used} evaluations, not 1")

## 6. A trivial search, to see the trajectory

Random sampling, 300 evaluations. This is not an algorithm you submit — it is
the baseline M0 asks you to beat.

In [ ]:
search = SensorPlacementProblem(instance, budget=5000, seed=0)
rng = np.random.default_rng(1)
for _ in range(300):
    a = rng.random(spec.n_candidates) < rng.uniform(0.1, 0.5)
    p = rng.uniform(spec.power_min, spec.power_max, spec.n_candidates)
    search.evaluate(a, p)

record = search.record
fig, ax = plt.subplots(figsize=(8, 3.2))
ax.plot(record.values, lw=0.6, alpha=0.5, label="each evaluation")
ax.plot(record.best_so_far, lw=2, label="best so far")
ax.set_xlabel("evaluation"); ax.set_ylabel("objective"); ax.legend(fontsize=8)
ax.set_title("300 random samples")
plt.tight_layout(); plt.show()

print(f"best found: {record.best:.4f} in {record.evaluations_used} evaluations")

## If everything above ran

Your environment is correct. Open `m0_foundations.ipynb` and set the same
`STUDENT_ID` and `SALT` there.

If something failed, the usual causes are: the ZIP was not unzipped, `sys.path`
does not point at `src`, or Python is older than 3.10.